In [ ]:
#!/usr/bin/env python3
"""
Download LPRM AMSR2 daily L3 files by matching the YYYYMMDD prefix
and saving the .nc4 file(s) present in the month directory.

Notes:
- GESDISC stores files under /{year}/{month}/ and filenames include a time stamp:
  e.g. LPRM-AMSR2_L3_D_SOILM3_V001_20150101005720.nc4
- This script finds filenames by scraping the month listing and matching the date.

username: ------
password: ------
"""
import os, re, datetime, getpass, pathlib, sys
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from http.cookiejar import CookieJar
import urllib.request
import netrc

BASE = "https://hydro1.gesdisc.eosdis.nasa.gov/data/WAOB/LPRM_AMSR2_D_SOILM3.001"

def get_credentials():
    u = os.environ.get("EARTHDATA_USERNAME")
    p = os.environ.get("EARTHDATA_PASSWORD")
    if u and p:
        return u, p
    try:
        rc = netrc.netrc()
        auth = rc.authenticators("urs.earthdata.nasa.gov")
        if auth:
            return auth[0], auth[2]
    except Exception:
        pass
    u = input("Earthdata username: ").strip()
    p = getpass.getpass("Earthdata password (hidden): ")
    if not u or not p:
        sys.exit("No credentials provided")
    return u, p

def make_session(user, pw):
    session = requests.Session()
    session.auth = (user, pw)
    retries = Retry(total=5, backoff_factor=1,
                    status_forcelist=[429, 500, 502, 503, 504])
    session.mount('https://', HTTPAdapter(max_retries=retries))
    return session

def urllib_cookie_open(user, pw):
    pm = urllib.request.HTTPPasswordMgrWithDefaultRealm()
    pm.add_password(None, "https://urs.earthdata.nasa.gov", user, pw)
    cj = CookieJar()
    opener = urllib.request.build_opener(
        urllib.request.HTTPBasicAuthHandler(pm),
        urllib.request.HTTPCookieProcessor(cj)
    )
    urllib.request.install_opener(opener)
    return opener

def list_month_files(session, year, month):
    """Return HTML of month index (or raise)."""
    url = f"{BASE}/{year}/{month:02d}/"
    r = session.get(url, timeout=30)
    r.raise_for_status()
    return r.text

def find_filenames_for_date(html, date_yyyymmdd):
    """
    Find filenames that start with the dataset prefix and the date:
    LPRM-AMSR2_L3_D_SOILM3_V001_{YYYYMMDD}*.nc4
    """
    pattern = re.compile(r"(LPRM-AMSR2_L3_D_SOILM3_V001_" + re.escape(date_yyyymmdd) + r"\d{6}\.nc4)")
    return sorted(set(pattern.findall(html)))

def download_with_requests(session, url, outpath):
    r = session.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(outpath, 'wb') as f:
        for chunk in r.iter_content(chunk_size=16_384):
            if chunk:
                f.write(chunk)

def download_with_urllib(opener, url, outpath):
    try:
        with opener.open(url, timeout=120) as resp, open(outpath, 'wb') as out:
            while True:
                chunk = resp.read(16384)
                if not chunk:
                    break
                out.write(chunk)
    except Exception as e:
        raise

def main():
    user, pw = get_credentials()
    session = make_session(user, pw)
    opener = urllib_cookie_open(user, pw)

    # adjust dates here
    start = datetime.date(2018, 3, 13)
    end   = datetime.date(2020, 12, 31)   # example: one-month test; change to your desired range

    cur = start
    while cur <= end:
        year = cur.year
        month = cur.month
        ymd = cur.strftime("%Y%m%d")
        print(f"\nProcessing {ymd}...")

        # Step 1: fetch month index
        try:
            html = list_month_files(session, year, month)
        except requests.HTTPError as e:
            print("requests failed to list month directory:", e)
            # try urllib fallback for listing (rare)
            try:
                idx_url = f"{BASE}/{year}/{month:02d}/"
                with opener.open(idx_url, timeout=30) as resp:
                    html = resp.read().decode('utf-8', errors='ignore')
            except Exception as e2:
                print("urllib fallback also failed for listing:", e2)
                cur += datetime.timedelta(days=1)
                continue

        # Step 2: find matching filenames
        files = find_filenames_for_date(html, ymd)
        if not files:
            print("No file found for", ymd)
            cur += datetime.timedelta(days=1)
            continue

        # Step 3: download each matching file (usually 1 per day for descending/ascending separation)
        for fname in files:
            url = f"{BASE}/{year}/{month:02d}/{fname}"
            #out = fname
            output_dir = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/lprm/data"
            os.makedirs(output_dir, exist_ok=True)  # create directory if missing
            out = os.path.join(output_dir, fname)

            if os.path.exists(out):
                print("Already exists:", out)
                continue
            print("Downloading", url)
            try:
                download_with_requests(session, url, out)
                print("Saved", out, "(via requests)")
            except Exception as e:
                print("requests download failed, trying urllib cookie fallback:", e)
                try:
                    download_with_urllib(opener, url, out)
                    print("Saved", out, "(via urllib cookie)")
                except Exception as e2:
                    print("Failed to download", url, ":", e2)

        cur += datetime.timedelta(days=1)


if __name__ == "__main__":
    main()



Processing 20180313...
Saved /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/lprm/data/LPRM-AMSR2_L3_D_SOILM3_V001_20180313001447.nc4 (via requests)

Processing 20180314...
Saved /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/lprm/data/LPRM-AMSR2_L3_D_SOILM3_V001_20180314005803.nc4 (via requests)

Processing 20180315...
Saved /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/lprm/data/LPRM-AMSR2_L3_D_SOILM3_V001_20180315000222.nc4 (via requests)

Processing 20180316...
Saved /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/lprm/data/LPRM-AMSR2_L3_D_SOILM3_V001_20180316004537.nc4 (via requests)

Processing 20180317...
Saved /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/lprm/data/LPRM-AMSR2_L3_D_SOILM3_V001_20180317012851.nc4 (via requests)

Processing 20180318...
Saved /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/lprm/data/LPRM-AMSR2_L3_D_SOILM3_V001_20180318003313.nc4 (via requests)

Processing 20180319..